# E5 · Re-ranking morfológico con lenguaje natural

**Objetivo.** Resolver empates visuales del E4 congelado sin alterar coincidencias visuales confiables. **Pregunta.** ¿Los atributos morfológicos observables recuperan consultas ambiguas dentro del Top-5? **Hipótesis.** Un doble umbral, calibrado sólo en validación, mejora Rank-1 sin dañar aciertos de E4.

La intervención es un re-ranker interpretable; E4 seed 46 (single + max) permanece fijo. Qwen3-VL se usa únicamente como extractor determinista de atributos sobre el recorte YOLO [Qwen3-VL, repositorio oficial](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct). El vocabulario, sus pesos y las dos compuertas son decisiones de esta investigación. Public v2 y PetTrace v6 nunca comparten galería.

**Protocolo.** La calibración usa exclusivamente Public v2 val. Después de congelar (tau_v, lambda, tau_t), cada test se evalúa una sola vez. Se reportan Rank-1/2/5 y mAP fotográfico; la compatibilidad por identidad es el máximo entre sus fotos de galería y el bono semántico se aplica a sus fotos del Top-5 sin cambiar su orden visual interno. Las descripciones no reciben rutas, nombres, identidad ni metadatos.


## Fase 1. Preparación y preprocesamiento de datos

Se instalan versiones exactas, se cargan credenciales desde Colab Secrets y se verifican los snapshots y el control E4 antes de leer métricas de test. Drive conserva únicamente el checkpoint de atributos para poder reanudar una sesión interrumpida.


In [ ]:
%pip install -q --no-cache-dir --upgrade "pettrace-ml[vlm,notebooks,tracking]==0.2.46"

import sys
if sys.platform != "linux" or not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("E5 requires Linux with Python 3.11–3.13")

import json, os, torch
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from IPython.display import Markdown, display

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before E5; a T4 uses the audited 2B fallback")


In [ ]:
from google.colab import drive, userdata

for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Missing Colab secret: {name}")
    os.environ[name] = value.strip()
os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
drive.mount("/content/drive")

LOG_TO_MLFLOW = True
DATASET_VERSION = "v6"
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
PUBLIC_VERSION = "v2"
MODEL_URI = "models:/pettrace-embedder/23"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = Path("reports") / f"e5-{run_id}"
PUBLIC_REPORT = REPORT / "public_test"
ATTR_CACHE = Path("/content/drive/MyDrive/PetTrace/checkpoints/e5/attributes-v3.parquet")
REPORT.mkdir(parents=True, exist_ok=True)


In [ ]:
from pettrace.data import training
from pettrace.evaluation.research import public_split_dataset, refined_public_receipt
from pettrace.experiments.e3 import load_benchmarks
from pettrace.experiments.e5 import load_frozen_e4

prepared = training(version=PUBLIC_VERSION, token=os.environ["HF_TOKEN"])
public_source = refined_public_receipt(prepared)
validation_ds = public_split_dataset(prepared, public_source, "val")
benchmarks = load_benchmarks(
    prepared, REPORT, private_version=DATASET_VERSION, private_commit=DATASET_COMMIT)
model, e4_control = load_frozen_e4(MODEL_URI, REPORT)
print(f"E4 frozen · validation {len(validation_ds.photos)} · Public {len(benchmarks.public.photos)} · PetTrace {len(benchmarks.private.photos)}")


## Fase 2. Representación y calibración

Qwen3-VL-2B permanece fijo para que cambiar de GPU no cambie el método. La revisión del modelo se fija a un commit de 40 caracteres. La decodificación es greedy (`do_sample=False`) y el caché Parquet es direccionado por SHA-256 de cada foto. Una respuesta con formato inválido se repara una vez de forma determinista; si vuelve a fallar, se conserva como atributos desconocidos y se contabiliza sin detener el lote. Los tests se describen sin usar sus etiquetas para seleccionar umbrales.


In [ ]:
from huggingface_hub import HfApi
from pettrace.evaluation.attributes import (
    MODEL_2B, QwenAttributeExtractor, align_attributes, attribute_cache_identity,
    extract_attribute_cache, load_attribute_cache,
)
from pettrace.evaluation.core import evaluate
from pettrace.experiments.e5 import select_calibration

attribute_model = MODEL_2B
attribute_revision = HfApi(token=os.environ["HF_TOKEN"]).model_info(attribute_model).sha
groups = {
    "public_validation": validation_ds.photos,
    "public_test": benchmarks.public.photos,
    "pettrace_v6": benchmarks.private.photos,
}
datasets = {
    "public": {"version": PUBLIC_VERSION, "commit": public_source["commit"]},
    "private": {"version": DATASET_VERSION, "commit": DATASET_COMMIT},
}
expected_identity = attribute_cache_identity(attribute_model, attribute_revision, datasets)
expected_rows = sum(len(photos) for photos in groups.values())
if ATTR_CACHE.is_file() and ATTR_CACHE.with_suffix(".receipt.json").is_file():
    attribute_receipt = json.loads(ATTR_CACHE.with_suffix(".receipt.json").read_text())
else:
    extractor = QwenAttributeExtractor(
        model_id=attribute_model, revision=attribute_revision, token=os.environ["HF_TOKEN"])
    attribute_receipt = extract_attribute_cache(
        groups, model.pipeline, extractor, ATTR_CACHE, datasets)
attribute_records, attribute_sha256 = load_attribute_cache(
    ATTR_CACHE, expected_sha256=attribute_receipt["sha256"],
    expected_identity=expected_identity)
if attribute_receipt["rows"] != expected_rows:
    raise RuntimeError("Attribute cache does not cover the frozen E5 population")
status_counts = attribute_receipt["status_counts"]
display(Markdown(
    f"**Attribute cache:** {attribute_receipt['rows']:,} photos · "
    f"ok {status_counts['ok']:,} · repaired {status_counts['repaired']:,} · "
    f"fallback {status_counts['fallback']:,} · SHA-256 `{attribute_sha256[:12]}…`"))

validation_visual = evaluate(validation_ds, model.pipeline, zero_shot=True)
validation_attributes = align_attributes(validation_visual.photos, attribute_records)
grid = product(
    (0.01, 0.02, 0.03, 0.05, 0.08, 0.10),
    (0.02, 0.05, 0.10, 0.20),
    (0.05, 0.10, 0.20, 0.30),
)
calibration = select_calibration(
    validation_visual, validation_attributes, grid, REPORT,
    attribute_cache_sha256=attribute_sha256)
display(Markdown(
    f"**Frozen on validation:** tau_v={calibration['tau_v']}, lambda={calibration['lambda']}, "
    f"tau_t={calibration['tau_t']} · Rank-1 {calibration['rank1']:.2%} · "
    f"damaged {calibration['damaged_queries']}"))


## Fase 3. Evaluación y análisis de resultados

Los umbrales ya están congelados. Se comprueba primero que E4 reproduce el baseline oficial y luego se comparan E4, E5 sin compuertas y E5 con doble compuerta. Esta es la única lectura de métricas de test. Toda corrida debe terminar con dos recibos MLflow.


In [ ]:
from pettrace.experiments.e5 import CONTROL_RESULTS, apply, as_evaluation

public_visual = evaluate(benchmarks.public, model.pipeline, zero_shot=True)
private_visual = evaluate(benchmarks.private, model.pipeline, zero_shot=True)
for visual, expected in ((public_visual, CONTROL_RESULTS["public"]), (private_visual, CONTROL_RESULTS["private"])):
    actual = visual.result["test"]["global"]
    if actual["queries"] != expected["queries"] or actual["gallery_animals"] != expected["identities"]:
        raise RuntimeError("E4 benchmark population differs from the frozen control")
    for key in ("rank1", "rank2", "mAP"):
        if abs(actual[key] - expected[key]) > 0.0006:
            raise RuntimeError(f"E4 {key} differs from the frozen control")

public_attributes = align_attributes(public_visual.photos, attribute_records)
private_attributes = align_attributes(private_visual.photos, attribute_records)
public_conditions, private_conditions = apply(
    public_visual, private_visual, public_attributes, private_attributes, calibration)
public_e5 = as_evaluation(public_visual, public_conditions["gated"])
private_e5 = as_evaluation(private_visual, private_conditions["gated"])


In [ ]:
from pettrace.evaluation.figures import save
from pettrace.experiments.e5 import save_comparison

save(public_e5, model, PUBLIC_REPORT, label="E5 · Public v2")
save(private_e5, model, REPORT, label="E5 · PetTrace v6")
comparison = save_comparison(public_conditions, private_conditions, REPORT)
display(Markdown(comparison))


In [ ]:
from pettrace.experiments.e5 import publish_results
from pettrace.tracking import source_identity

if not LOG_TO_MLFLOW:
    raise RuntimeError("MLflow publication is mandatory for E5")
publication = publish_results(
    public_e5, private_e5, model, REPORT, PUBLIC_REPORT,
    source_identity()["source_sha256"], calibration=calibration,
    attribute_receipt=attribute_receipt)
if not publication.get("published"):
    raise RuntimeError("E5 is incomplete: MLflow publication did not finish")
display(Markdown(
    f"**E5 published.** Public run {publication['public_run_id']} · "
    f"PetTrace run [{publication['run_id']}]({publication['url']})"))


**Limitaciones.** El VLM aporta etiquetas discretas, no verdad morfológica; sus errores pueden correlacionarse con pose, iluminación y especie. La compuerta reduce daño, no demuestra generalización abierta ni separación por sesión. La EER continúa siendo descriptiva sobre los mismos pares. El barrido estima sensibilidad en una sola validación y no es incertidumbre poblacional. No se promovió ningún alias de serving.
